# 02 · Zaman Serisi: Durağanlık, Sahte Regresyon ve Eşbütünleşme

Zaman serisi verisiyle regresyon kurmadan önce sorulması gereken soru: **seriler durağan mı?** Durağan olmayan iki seri arasında gerçekte hiçbir ilişki yokken bile yüksek R² ve "anlamlı" katsayılar çıkabilir (**sahte regresyon**, Granger ve Newbold, 1974).

**Öğreneceklerin**
1. ADF ve KPSS testleriyle durağanlığı sınamak ve iki testi birlikte okumak
2. Bütünleşme derecesine (I(0) / I(1)) karar vermek ve bunu tez tablosu olarak raporlamak
3. Sahte regresyonu kendi gözünle görmek
4. Engle-Granger testiyle eşbütünleşmeyi (uzun dönem ilişkisini) sınamak

> ⚠️ Kullanılan veriler **simüle edilmiştir**.

In [1]:
import numpy as np
import pandas as pd
import ekonometri as ek

veri = ek.veri.yukle("ihracat")
veri.head()

,ihracat,reel_kur,dunya_buyumesi,ceyrek
donem,,,,
2005Q1,15.40,107.84,4.88,1
2005Q2,15.93,99.00,3.95,2
2005Q3,15.66,100.87,3.95,3
2005Q4,16.06,99.28,4.34,4
2006Q1,15.62,98.11,3.55,1


## 1. Tek bir seri: ADF ve KPSS

- **ADF**: H0 = seride birim kök var (durağan değil). Reddedersen seri durağandır.
- **KPSS**: H0 = seri durağan. Reddedersen seri durağan değildir.

İki testin hipotezleri **ters** olduğu için birlikte kullanılır: ikisi aynı sonuca işaret ediyorsa karar güçlüdür.

In [2]:
ek.adf(np.log(veri["reel_kur"]), ad="ln(reel_kur)")

⚠️ ADF — 'ln(reel_kur)' (sabitli, gecikme = 0, n = 79)
   istatistik = -1,768, p = 0,3963   kritik değerler: %1: -3,516 · %5: -2,899 · %10: -2,587
   H0: Seride birim kök vardır (seri durağan değildir).
   p ≥ 0,05: H0 reddedilemedi — 'ln(reel_kur)' serisinde birim kök var (durağan değil). Birinci farkını test et.

In [3]:
ek.kpss(np.log(veri["reel_kur"]), ad="ln(reel_kur)")

⚠️ KPSS — 'ln(reel_kur)' (sabitli, gecikme = 5, n = 80)
   istatistik = 0,669, p = 0,0164   kritik değerler: %1: 0,739 · %2,5: 0,574 · %5: 0,463 · %10: 0,347
   H0: Seri durağandır.
   p < 0,05: H0 reddedildi — 'ln(reel_kur)' serisi durağan değil.

## 2. Tez tablosu: tüm değişkenler, düzey ve birinci fark

`ek.duraganlik()` her değişken için düzeyde ve birinci farkta ADF + KPSS uygular ve bütünleşme derecesine karar verir. Gecikme sayısı AIC ile seçilir (köşeli parantez içinde).

In [4]:
d = ek.duraganlik(veri, ["ihracat", "reel_kur", "dunya_buyumesi"], log=["ihracat", "reel_kur"])
d

Değişken,ADF (düzey),p,KPSS (düzey),ADF (Δ),p,KPSS (Δ),Karar
ln(ihracat),"-1,341 [7]","0,610","1,405**","-6,651*** [6]","<0,001","0,177",I(1)
ln(reel_kur),"-1,768 [0]","0,396","0,669**","-10,482*** [0]","<0,001","0,132",I(1)
dunya_buyumesi,"-6,235*** [0]","<0,001","0,080","-8,650*** [2]","<0,001","0,263",I(0)


In [5]:
print(d.rapor().paragraflar[0])

Değişkenlerin durağanlık özellikleri ADF (Dickey ve Fuller, 1979) ve KPSS (Kwiatkowski vd., 1992) testleriyle incelenmiş, sonuçlar Tablo 1 ile sunulmuştur. ADF testine göre 'dunya_buyumesi' değişkeni düzeyde durağandır; dolayısıyla I(0) sürecidir. 'ln(ihracat)', 'ln(reel_kur)' değişkenleri düzeyde birim kök içermekte, birinci farkları alındığında durağan hâle gelmektedir; dolayısıyla I(1) süreçleridir. KPSS testi sonuçları ADF testi sonuçlarını desteklemektedir. Değişkenlerin bir kısmının I(0), bir kısmının I(1) olması ve hiçbirinin I(2) olmaması, uzun dönem ilişkisinin ARDL sınır testi gibi bütünleşme derecesine esnek yaklaşımlarla incelenmesine uygundur.


Bu bölüm Word'e doğrudan aktarılabilir: `d.rapor("duraganlik.docx")`.

## 3. Sahte regresyon deneyi

Birbirinden **tamamen bağımsız** iki rastgele yürüyüş üretelim ve birini diğerine regresleyelim:

In [6]:
rng = np.random.default_rng(1)
sahte = pd.DataFrame({"a": np.cumsum(rng.normal(size=200)), "b": np.cumsum(rng.normal(size=200))})
m = ek.ekk("a ~ b", sahte)
print(f"R² = {m.r2:.2f},  b katsayısının p-değeri = {m.p_degerleri['b']:.2g}")

R² = 0.73,  b katsayısının p-değeri = 6.9e-58


Aralarında hiçbir ilişki olmadığı hâlde katsayı "anlamlı" görünebilir. Engle-Granger testi bu tuzağı yakalar:

In [7]:
ek.engle_granger("a ~ b", sahte)

⚠️ Engle-Granger eşbütünleşme testi (2 değişken)
   Kalıntılara ADF: τ = -1,441, p = 0,7830 (gecikme = 2)   kritik değerler: %1: -3,952 · %5: -3,367 · %10: -3,066
   H0: Değişkenler arasında eşbütünleşme yoktur.
   p ≥ 0,05: H0 reddedilemedi — eşbütünleşme yok: düzey değişkenlerle kurulan regresyon sahte (spurious) olabilir.

## 4. Gerçek bir uzun dönem ilişkisi

Şimdi ortak bir stokastik trendi paylaşan iki seri üretelim: `y = 2 + 0,8·x + durağan hata`. Seriler tek tek durağan değil ama aralarındaki fark durağan: **eşbütünleşik**.

In [8]:
x = np.cumsum(rng.normal(size=200))
es = pd.DataFrame({"x": x, "y": 2 + 0.8 * x + rng.normal(size=200)})
eg = ek.engle_granger("y ~ x", es)
eg

✅ Engle-Granger eşbütünleşme testi (2 değişken)
   Kalıntılara ADF: τ = -13,377, p < 0,0001 (gecikme = 0)   kritik değerler: %1: -3,952 · %5: -3,367 · %10: -3,066
   H0: Değişkenler arasında eşbütünleşme yoktur.
   p < 0,05: H0 reddedildi — değişkenler eşbütünleşik: aralarında uzun dönemli bir denge ilişkisi var.

In [9]:
print(eg.rapor().paragraflar[0])

Değişkenler arasındaki uzun dönemli ilişki Engle ve Granger (1987) iki aşamalı yöntemiyle sınanmıştır. İlk aşamada uzun dönem denklemi EKK ile tahmin edilmiş (R² = 0,963), ikinci aşamada bu denklemin kalıntılarına sabitsiz ADF testi uygulanmıştır. Kalıntılara uygulanan ADF test istatistiği -13,377 olarak hesaplanmıştır (p < 0,001; %5 kritik değeri -3,367). Bu sonuca göre eşbütünleşme olmadığını ifade eden temel hipotez %5 düzeyinde reddedilmektedir; değişkenler eşbütünleşiktir ve aralarında uzun dönemli bir denge ilişkisi bulunmaktadır. Uzun dönem katsayıları: x = 0,7777. Bu katsayıların standart hataları standart dağılımlara uymadığından anlamlılıkları t-istatistikleriyle değerlendirilmemelidir.
